# CP5 · FIAP Bank · Anonimização de dados para exportação externa (LGPD)

**Aluno:** Murilo Benhossi · RM 562358 · Tecnólogo em Inteligência Artificial · RPA aplicado com IA

Pipeline completo, do CSV até a persistência das versões anonimizadas:

| Etapa | O que faz | Onde |
|---|---|---|
| a | Lê `data/manifestacoes_clientes_cp5.csv` | Seção 3 |
| b | Extrai campos estruturados via LLM com `response_schema` / Pydantic | Seções 4 e 5 |
| c | Retry com backoff, throttling, fallback Gemini → Groq, cache e idempotência | Seção 5 |
| d | Persiste texto, resumo e campos no SQLite | Seções 2 e 6 |
| e, f | Gera e persiste `texto_anonimizado` e `resumo_anonimizado` sem sobrescrever os originais | Seção 7 |
| g | Auditoria antes/depois, recall e precisão a partir de rotulagem manual | Seção 8 |
| — | Governança | Seção 9 |

## 0. Instalação
No Colab, rode a célula abaixo uma vez. Localmente, use `pip install -r requirements.txt` (ver README).

In [1]:
import sys, subprocess, os
from pathlib import Path

EM_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/murilo2318/cp5-anonimizacao-lgpd.git"  # ajuste se o nome do repo for outro

if EM_COLAB:
    if not Path("anonimizar_texto.py").exists():
        subprocess.run(["git", "clone", "-q", REPO_URL, "repo"], check=True)
        os.chdir("repo")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "google-genai", "groq", "pydantic", "spacy"], check=True)
    subprocess.run([sys.executable, "-m", "spacy", "download", "pt_core_news_lg"], check=True)
print("Diretorio de trabalho:", os.getcwd())

Diretorio de trabalho: /Users/mur_benhs/Downloads/cp5-anonimizacao-lgpd


## 1. Configuração e chaves
As chaves **nunca** ficam no código. No Colab vêm do painel *Secrets* (`userdata.get`); localmente, de variáveis de ambiente `GEMINI_API_KEY` e `GROQ_API_KEY` (ou de um arquivo `.env`, que está no `.gitignore`).

In [2]:
import os, json, time, random, hashlib, sqlite3, logging
from datetime import datetime, timezone
from pathlib import Path

import pandas as pd

logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s %(message)s", force=True)
log = logging.getLogger("cp5")


def ler_segredo(nome: str) -> str | None:
    try:
        from google.colab import userdata  # type: ignore
        valor = userdata.get(nome)
        if valor:
            return valor
    except Exception:
        pass
    if Path(".env").exists():  # leitura simples de .env local, sem dependencia extra
        for linha in Path(".env").read_text(encoding="utf-8").splitlines():
            if linha.strip().startswith(nome + "="):
                os.environ.setdefault(nome, linha.split("=", 1)[1].strip().strip('"').strip("'"))
    return os.environ.get(nome)


GEMINI_API_KEY = ler_segredo("GEMINI_API_KEY")
GROQ_API_KEY = ler_segredo("GROQ_API_KEY")
print("Gemini key carregada:", bool(GEMINI_API_KEY), "| Groq key carregada:", bool(GROQ_API_KEY))

# Modelos free-tier (confira os nomes atuais no AI Studio e no console do Groq)
MODELO_GEMINI = os.environ.get("MODELO_GEMINI", "gemini-3.5-flash")
MODELO_GROQ = os.environ.get("MODELO_GROQ", "openai/gpt-oss-120b")

CSV_PATH = Path("data/manifestacoes_clientes_cp5.csv")
DB_PATH = Path("fiap_bank_cp5.db")

# Throttling proativo: intervalo minimo entre chamadas de cada provedor (s).
# Gemini free-tier ~10 RPM -> 6,5 s; Groq free-tier ~30 RPM -> 2,5 s.
INTERVALO_MIN = {"gemini": 6.5, "groq": 2.5}
MAX_TENTATIVAS = 4
BACKOFF_BASE = 2.0  # espera = base ** tentativa + jitter

# Demonstracao obrigatoria do fallback (secao 4a do enunciado):
# para os ids abaixo o provedor principal falha de proposito ANTES da chamada
# de rede, simulando indisponibilidade. O pipeline tem que cair no Groq.
DEMONSTRAR_FALLBACK = True
IDS_FALHA_FORCADA_GEMINI = {str(i) for i in range(1, 41) if i % 3 == 0}  # 13 de 40 ids

Gemini key carregada: True | Groq key carregada: True


## 2. Banco SQLite
A tabela base é a mesma das aulas anteriores (texto, campos extraídos, resumo). As colunas novas do CP5 (`texto_anonimizado`, `resumo_anonimizado`, `provedor`) entram por **migração** com `ALTER TABLE`, só se ainda não existirem. Todas as queries são parametrizadas (`?`).

In [3]:
def conectar(caminho: Path = DB_PATH) -> sqlite3.Connection:
    con = sqlite3.connect(caminho)
    con.row_factory = sqlite3.Row
    return con


def criar_schema(con: sqlite3.Connection) -> None:
    con.execute(
        '''CREATE TABLE IF NOT EXISTS manifestacoes (
               id                TEXT PRIMARY KEY,
               texto             TEXT NOT NULL,
               hash_texto        TEXT NOT NULL,
               tipo_manifestacao TEXT,
               sentimento        TEXT,
               valor_citado      REAL,
               urgencia          TEXT,
               resumo            TEXT,
               status            TEXT NOT NULL,          -- processado | revisao_humana | falha
               motivo_revisao    TEXT,
               processado_em     TEXT
           )'''
    )
    con.execute(
        '''CREATE TABLE IF NOT EXISTS cache_llm (
               hash_texto   TEXT PRIMARY KEY,
               provedor     TEXT NOT NULL,
               resposta     TEXT NOT NULL,
               criado_em    TEXT NOT NULL
           )'''
    )
    # Migracao CP5: colunas novas, sem apagar nada do que ja existe
    existentes = {r["name"] for r in con.execute("PRAGMA table_info(manifestacoes)")}
    for coluna in ("texto_anonimizado", "resumo_anonimizado", "provedor"):
        if coluna not in existentes:
            # nome de coluna nao aceita placeholder; vem de uma tupla fixa acima
            con.execute(f"ALTER TABLE manifestacoes ADD COLUMN {coluna} TEXT")
            log.info("Coluna adicionada: %s", coluna)
    con.commit()


con = conectar()
criar_schema(con)
pd.read_sql_query("PRAGMA table_info(manifestacoes)", con)[["name", "type"]]

,name,type
0,id,TEXT
1,texto,TEXT
2,hash_texto,TEXT
3,tipo_manifestacao,TEXT
4,sentimento,TEXT
5,valor_citado,REAL
6,urgencia,TEXT
7,resumo,TEXT
8,status,TEXT
9,motivo_revisao,TEXT


## 3. Leitura do CSV

In [4]:
df = pd.read_csv(CSV_PATH, dtype={"id": str})
print(df.shape)
df.head()

(40, 2)


,id,texto
0,1,"Fui cobrado duas vezes na fatura deste mes, o ..."
1,2,Gostaria de saber como altero meu endereco de ...
2,3,Meu nome e Carlos Eduardo Ferreira e identifiq...
3,4,"O atendimento da agencia foi otimo, o gerente ..."
4,5,Recebi uma mensagem estranha pedindo meus dado...


## 4. Schema de extração (Pydantic + `response_schema`)
Mesmo modelo da Aula 12. O Gemini recebe a classe no `response_schema` e devolve `resposta.parsed`; o Groq recebe o JSON Schema no prompt e a resposta é validada com o mesmo modelo.

In [5]:
from typing import Literal, Optional
from pydantic import BaseModel, Field, ValidationError


class ManifestacaoExtraida(BaseModel):
    tipo_manifestacao: Literal["reclamacao", "solicitacao", "duvida", "elogio", "sugestao", "fraude"]
    sentimento: Literal["positivo", "neutro", "negativo"]
    valor_citado: Optional[float] = Field(None, description="Valor em reais citado no texto, ou null")
    urgencia: Literal["baixa", "media", "alta", "critica"]
    resumo: str = Field(..., description="Resumo objetivo em ate 25 palavras")


PROMPT_EXTRACAO = '''Voce e o robo de triagem do FIAP Bank. Leia a manifestacao do cliente e extraia:
- tipo_manifestacao: reclamacao, solicitacao, duvida, elogio, sugestao ou fraude
- sentimento: positivo, neutro ou negativo
- valor_citado: valor em reais mencionado, ou null
- urgencia: baixa, media, alta ou critica (fraude e perda de acesso sao alta/critica)
- resumo: ate 25 palavras, em portugues

Manifestacao:
<manifestacao>{texto}</manifestacao>'''

SCHEMA_JSON = json.dumps(ManifestacaoExtraida.model_json_schema(), ensure_ascii=False)

## 5. Pilha de resiliência (Aula 11)

* **Throttling proativo** (`Throttle`): garante o intervalo mínimo entre chamadas de cada provedor, antes de estourar a cota.
* **Retry com backoff exponencial + jitter** (`com_retry`): só para erros transitórios (429, 5xx, timeout, conexão). Erro permanente (chave inválida, 400) não é repetido, vai direto para o fallback.
* **Fallback Gemini → Groq** (`extrair_com_fallback`).
* **Cache** (`cache_llm`, chaveado pelo hash do texto): texto idêntico não gera nova chamada.
* **Idempotência** (`ja_processada`): consulta o banco antes de chamar a API; manifestação já persistida não é reprocessada.
* **Revisão humana**: `parsed is None` (Gemini) ou JSON fora do schema (Groq) não quebra o pipeline; a linha é gravada com `status='revisao_humana'`.

In [6]:
from google import genai
from google.genai import types, errors as gerrors
import groq as groq_sdk

cliente_gemini = genai.Client(api_key=GEMINI_API_KEY) if GEMINI_API_KEY else None
cliente_groq = groq_sdk.Groq(api_key=GROQ_API_KEY) if GROQ_API_KEY else None


class ErroTransitorio(Exception):
    '''Vale a pena tentar de novo (cota, 5xx, rede).'''


class ErroPermanente(Exception):
    '''Nao adianta repetir (chave invalida, requisicao invalida, falha forcada).'''


class RespostaForaDoSchema(Exception):
    '''Provedor respondeu, mas sem objeto valido -> revisao humana.'''

    def __init__(self, provedor: str, trecho: str):
        super().__init__(trecho)
        self.provedor = provedor


class Throttle:
    def __init__(self, intervalos: dict[str, float]):
        self.intervalos = intervalos
        self.ultima: dict[str, float] = {}

    def aguardar(self, provedor: str) -> None:
        espera = self.intervalos[provedor] - (time.monotonic() - self.ultima.get(provedor, 0.0))
        if espera > 0:
            time.sleep(espera)
        self.ultima[provedor] = time.monotonic()


throttle = Throttle(INTERVALO_MIN)
CONTADOR_CHAMADAS = {"gemini": 0, "groq": 0}


def com_retry(func, provedor: str, *args, **kwargs):
    for tentativa in range(1, MAX_TENTATIVAS + 1):
        throttle.aguardar(provedor)
        try:
            return func(*args, **kwargs)
        except ErroTransitorio as e:
            if tentativa == MAX_TENTATIVAS:
                raise ErroPermanente(f"{provedor}: esgotou {MAX_TENTATIVAS} tentativas ({e})") from e
            espera = BACKOFF_BASE ** tentativa + random.uniform(0, 1)
            log.warning("%s transitorio (%s). Tentativa %d/%d, aguardando %.1fs", provedor, e, tentativa, MAX_TENTATIVAS, espera)
            time.sleep(espera)


def chamar_gemini(id_: str, texto: str) -> ManifestacaoExtraida:
    if DEMONSTRAR_FALLBACK and id_ in IDS_FALHA_FORCADA_GEMINI:
        raise ErroPermanente("falha forcada do Gemini (demonstracao de fallback)")
    if cliente_gemini is None:
        raise ErroPermanente("GEMINI_API_KEY ausente")
    CONTADOR_CHAMADAS["gemini"] += 1
    try:
        resposta = cliente_gemini.models.generate_content(
            model=MODELO_GEMINI,
            contents=PROMPT_EXTRACAO.format(texto=texto),
            config=types.GenerateContentConfig(
                response_mime_type="application/json",
                response_schema=ManifestacaoExtraida,
                temperature=0,
            ),
        )
    except gerrors.APIError as e:
        if getattr(e, "code", None) in (429, 500, 502, 503, 504):
            raise ErroTransitorio(f"HTTP {e.code}") from e
        raise ErroPermanente(f"HTTP {getattr(e, 'code', '?')}: {e}") from e
    except (TimeoutError, ConnectionError) as e:
        raise ErroTransitorio(str(e)) from e

    if resposta.parsed is None:  # truncada ou fora do schema (Aula 12)
        raise RespostaForaDoSchema("gemini", (resposta.text or "")[:300])
    return resposta.parsed


def chamar_groq(id_: str, texto: str) -> ManifestacaoExtraida:
    if cliente_groq is None:
        raise ErroPermanente("GROQ_API_KEY ausente")
    CONTADOR_CHAMADAS["groq"] += 1
    try:
        resp = cliente_groq.chat.completions.create(
            model=MODELO_GROQ,
            temperature=0,
            response_format={"type": "json_object"},
            messages=[
                {"role": "system", "content": "Responda somente com um objeto JSON valido neste schema: " + SCHEMA_JSON},
                {"role": "user", "content": PROMPT_EXTRACAO.format(texto=texto)},
            ],
        )
    except (groq_sdk.RateLimitError, groq_sdk.InternalServerError, groq_sdk.APITimeoutError, groq_sdk.APIConnectionError) as e:
        raise ErroTransitorio(type(e).__name__) from e
    except groq_sdk.APIStatusError as e:
        raise ErroPermanente(f"HTTP {e.status_code}") from e

    conteudo = resp.choices[0].message.content or ""
    try:
        return ManifestacaoExtraida.model_validate_json(conteudo)
    except ValidationError as e:
        raise RespostaForaDoSchema("groq", conteudo[:300]) from e


def extrair_com_fallback(id_: str, texto: str) -> tuple[ManifestacaoExtraida, str]:
    try:
        return com_retry(chamar_gemini, "gemini", id_, texto), "gemini"
    except ErroPermanente as e:
        log.warning("id=%s Gemini indisponivel (%s). Fallback -> Groq", id_, e)
    return com_retry(chamar_groq, "groq", id_, texto), "groq"

In [7]:
def hash_texto(texto: str) -> str:
    return hashlib.sha256(texto.strip().encode("utf-8")).hexdigest()


def ja_processada(con, id_: str) -> bool:
    '''Idempotencia: consulta o banco antes de gastar cota.'''
    linha = con.execute(
        "SELECT status FROM manifestacoes WHERE id = ? AND status IN ('processado', 'revisao_humana')", (id_,)
    ).fetchone()
    return linha is not None


def buscar_cache(con, h: str):
    linha = con.execute("SELECT provedor, resposta FROM cache_llm WHERE hash_texto = ?", (h,)).fetchone()
    if linha:
        return ManifestacaoExtraida.model_validate_json(linha["resposta"]), linha["provedor"]
    return None


def gravar_cache(con, h: str, dados: ManifestacaoExtraida, provedor: str) -> None:
    con.execute(
        "INSERT OR IGNORE INTO cache_llm (hash_texto, provedor, resposta, criado_em) VALUES (?, ?, ?, ?)",
        (h, provedor, dados.model_dump_json(), datetime.now(timezone.utc).isoformat()),
    )


def gravar_manifestacao(con, id_, texto, h, dados, provedor, status, motivo=None) -> None:
    d = dados.model_dump() if dados else {}
    con.execute(
        '''INSERT INTO manifestacoes
               (id, texto, hash_texto, tipo_manifestacao, sentimento, valor_citado, urgencia, resumo,
                status, motivo_revisao, provedor, processado_em)
           VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
           ON CONFLICT(id) DO UPDATE SET
               tipo_manifestacao = excluded.tipo_manifestacao, sentimento = excluded.sentimento,
               valor_citado = excluded.valor_citado, urgencia = excluded.urgencia, resumo = excluded.resumo,
               status = excluded.status, motivo_revisao = excluded.motivo_revisao,
               provedor = excluded.provedor, processado_em = excluded.processado_em''',
        (id_, texto, h, d.get("tipo_manifestacao"), d.get("sentimento"), d.get("valor_citado"),
         d.get("urgencia"), d.get("resumo"), status, motivo, provedor,
         datetime.now(timezone.utc).isoformat()),
    )


def processar_lote(con, df: pd.DataFrame) -> dict:
    stats = {"pulados_idempotencia": 0, "cache": 0, "gemini": 0, "groq": 0, "revisao_humana": 0, "falha": 0}
    for id_, texto in zip(df["id"], df["texto"]):
        if ja_processada(con, id_):
            stats["pulados_idempotencia"] += 1
            continue
        h = hash_texto(texto)
        em_cache = buscar_cache(con, h)
        if em_cache:
            dados, provedor = em_cache
            gravar_manifestacao(con, id_, texto, h, dados, provedor, "processado")
            stats["cache"] += 1
        else:
            try:
                dados, provedor = extrair_com_fallback(id_, texto)
                gravar_cache(con, h, dados, provedor)
                gravar_manifestacao(con, id_, texto, h, dados, provedor, "processado")
                stats[provedor] += 1
            except RespostaForaDoSchema as e:
                log.error("id=%s resposta fora do schema -> revisao humana", id_)
                gravar_manifestacao(con, id_, texto, h, None, e.provedor, "revisao_humana", f"parsed is None: {e}")
                stats["revisao_humana"] += 1
            except ErroPermanente as e:
                log.error("id=%s falhou nos dois provedores: %s", id_, e)
                gravar_manifestacao(con, id_, texto, h, None, None, "falha", str(e))
                stats["falha"] += 1
        con.commit()  # commit por linha: se cair no meio, o que ja foi pago fica salvo
    return stats

### 5.1 Primeira execução (com falha forçada do Gemini em parte do lote)
Se o banco já existir, esta célula não chama nenhuma API (idempotência) e mostra todos os ids como pulados. A evidência do processamento original fica no próprio banco (seção 5.2, coluna `processado_em`).

In [8]:
inicio = time.time()
stats_1 = processar_lote(con, df)
print(stats_1, "| chamadas de API:", CONTADOR_CHAMADAS, f"| {time.time() - inicio:.0f}s")

{'pulados_idempotencia': 40, 'cache': 0, 'gemini': 0, 'groq': 0, 'revisao_humana': 0, 'falha': 0} | chamadas de API: {'gemini': 0, 'groq': 0} | 0s


### 5.2 Evidência do fallback no próprio banco
A coluna `provedor` precisa ter uma mistura real de `gemini` e `groq`.

In [9]:
pd.read_sql_query(
    "SELECT provedor, COUNT(*) AS qtd FROM manifestacoes GROUP BY provedor ORDER BY provedor", con
)

,provedor,qtd
0,gemini,12
1,groq,28


In [10]:
# Groq por falha forcada x Groq por falha real do Gemini (ex.: 503 de sobrecarga)
todos_groq = pd.read_sql_query(
    "SELECT id FROM manifestacoes WHERE provedor = ? ORDER BY CAST(id AS INTEGER)", con, params=("groq",)
)["id"].tolist()
forcados = [i for i in todos_groq if i in IDS_FALHA_FORCADA_GEMINI]
reais = [i for i in todos_groq if i not in IDS_FALHA_FORCADA_GEMINI]
print(f"Groq por falha forcada: {len(forcados)} | Groq por falha real do Gemini: {len(reais)} -> ids {reais}")

# Os ids em que o Gemini foi derrubado devem aparecer com provedor = 'groq'
placeholders = ",".join("?" * len(IDS_FALHA_FORCADA_GEMINI))
pd.read_sql_query(
    f"SELECT id, provedor, status FROM manifestacoes WHERE id IN ({placeholders}) ORDER BY CAST(id AS INTEGER)",
    con, params=sorted(IDS_FALHA_FORCADA_GEMINI, key=int),
)

Groq por falha forcada: 13 | Groq por falha real do Gemini: 15 -> ids ['19', '20', '22', '23', '25', '26', '28', '29', '31', '32', '34', '35', '37', '38', '40']


,id,provedor,status
0,3,groq,processado
1,6,groq,processado
2,9,groq,processado
3,12,groq,processado
4,15,groq,processado
5,18,groq,processado
6,21,groq,processado
7,24,groq,processado
8,27,groq,processado
9,30,groq,processado


### 5.3 Segunda execução: idempotência
Rodar de novo sobre o mesmo CSV não pode duplicar linhas nem chamar a API.

In [11]:
antes = dict(CONTADOR_CHAMADAS)
stats_2 = processar_lote(con, df)
print(stats_2)
print("Chamadas novas de API:", {k: CONTADOR_CHAMADAS[k] - antes[k] for k in antes})
print("Linhas na tabela:", con.execute("SELECT COUNT(*) FROM manifestacoes").fetchone()[0], "| linhas no CSV:", len(df))
assert all(CONTADOR_CHAMADAS[k] == antes[k] for k in antes), "segunda execucao gastou cota"
assert con.execute("SELECT COUNT(*) FROM manifestacoes").fetchone()[0] == df["id"].nunique()

{'pulados_idempotencia': 40, 'cache': 0, 'gemini': 0, 'groq': 0, 'revisao_humana': 0, 'falha': 0}
Chamadas novas de API: {'gemini': 0, 'groq': 0}
Linhas na tabela: 40 | linhas no CSV: 40


### 5.4 Fila de revisão humana
Linhas em que o LLM não devolveu objeto válido (`parsed is None`) ou em que os dois provedores falharam.

In [12]:
pd.read_sql_query(
    "SELECT id, status, motivo_revisao FROM manifestacoes WHERE status <> 'processado' ORDER BY CAST(id AS INTEGER)", con
)

,id,status,motivo_revisao


## 6. Campos extraídos persistidos

In [13]:
pd.read_sql_query(
    '''SELECT id, tipo_manifestacao, sentimento, valor_citado, urgencia, provedor, resumo
       FROM manifestacoes ORDER BY CAST(id AS INTEGER)''', con
)

,id,tipo_manifestacao,sentimento,valor_citado,urgencia,provedor,resumo
0,1,reclamacao,negativo,NaN,alta,gemini,Cliente reclama de cobranca duplicada na fatur...
1,2,duvida,neutro,NaN,baixa,gemini,Cliente tem dúvida sobre como alterar o endere...
2,3,reclamacao,negativo,NaN,media,groq,Cliente Carlos Eduardo Ferreira relata transaç...
3,4,elogio,positivo,NaN,baixa,gemini,Cliente elogia o otimo atendimento da agencia ...
4,5,fraude,negativo,NaN,alta,gemini,Cliente relata recebimento de mensagem suspeit...
5,6,duvida,neutro,NaN,baixa,groq,Cliente pergunta quando será lançada a próxima...
6,7,fraude,negativo,NaN,critica,gemini,Cliente Fernanda Albuquerque relata que seu ca...
7,8,elogio,positivo,NaN,baixa,gemini,Cliente elogia a rapidez na abertura da conta ...
8,9,reclamacao,negativo,NaN,media,groq,Cliente reclama da taxa de manutenção cobrada ...
9,10,duvida,neutro,NaN,baixa,gemini,Cliente tem dúvida sobre o funcionamento do pr...


## 7. Anonimização local (spaCy + regex)

A função está no arquivo `anonimizar_texto.py`, na raiz do repositório, com a assinatura fixa `anonimizar_texto(texto: str) -> str`. Ela trata os dois problemas como complementares:

* **Identificadores estruturados** (CPF, RG, conta/agência, e-mail): expressões regulares por formato e, quando o formato é ambíguo, pela palavra-chave anterior (`RG 501987654` vira `[RG]`; `conta 1234` vira `[CONTA]`; `cartão final 4521` fica intacto).
* **Nomes de pessoa**: NER do spaCy (`PER`), com filtro de falsos positivos (verbos no início de frase que o modelo marca como pessoa, como “Fui”, “Achei”) e gatilhos de contexto (“meu nome é”, “gerente”, “atendente”, “Att,”, “Assinado,”) para recuperar nomes em minúsculas ou rotulados como LOC/ORG.

Aplicada em `texto` e em `resumo`, gravando em colunas próprias. As colunas originais não são alteradas. A etapa também é idempotente: só processa linhas com `texto_anonimizado IS NULL`.

In [14]:
import importlib
import anonimizar_texto as anon
importlib.reload(anon)
from anonimizar_texto import anonimizar_texto

print("Modelo spaCy em uso:", anon._carregar_spacy().meta["name"] if anon._carregar_spacy() else "nenhum (fallback regex)")

pendentes = con.execute(
    "SELECT id, texto, resumo FROM manifestacoes WHERE texto_anonimizado IS NULL"
).fetchall()
for linha in pendentes:
    con.execute(
        "UPDATE manifestacoes SET texto_anonimizado = ?, resumo_anonimizado = ? WHERE id = ?",
        (anonimizar_texto(linha["texto"]),
         anonimizar_texto(linha["resumo"]) if linha["resumo"] else None,
         linha["id"]),
    )
con.commit()
print(f"{len(pendentes)} linhas anonimizadas nesta execucao")

Modelo spaCy em uso: core_news_lg
0 linhas anonimizadas nesta execucao


In [15]:
pd.set_option("display.max_colwidth", 120)
pd.read_sql_query(
    '''SELECT id, texto, texto_anonimizado, resumo, resumo_anonimizado
       FROM manifestacoes WHERE texto <> texto_anonimizado ORDER BY CAST(id AS INTEGER)''', con
)

,id,texto,texto_anonimizado,resumo,resumo_anonimizado
0,3,Meu nome e Carlos Eduardo Ferreira e identifiquei uma transacao que nao reconheco no meu extrato de hoje.,Meu nome e [NOME] e identifiquei uma transacao que nao reconheco no meu extrato de hoje.,"Cliente Carlos Eduardo Ferreira relata transação desconhecida no extrato de hoje, solicitando esclarecimento.","Cliente [NOME] relata transação desconhecida no extrato de hoje, solicitando esclarecimento."
1,4,"O atendimento da agencia foi otimo, o gerente Bruno Salviano resolveu tudo rapido.","O atendimento da agencia foi otimo, o gerente [NOME] resolveu tudo rapido.",Cliente elogia o otimo atendimento da agencia e a rapidez do gerente Bruno Salviano na resolucao de problemas.,Cliente elogia o otimo atendimento da agencia e a rapidez do gerente [NOME] na resolucao de problemas.
2,5,"Recebi uma mensagem estranha pedindo meus dados bancarios, acho que e uma fraude. Meu CPF e 123.456.789-00.","Recebi uma mensagem estranha pedindo meus dados bancarios, acho que e uma fraude. Meu CPF e [CPF].",Cliente relata recebimento de mensagem suspeita solicitando dados bancários e suspeita de tentativa de fraude.,Cliente relata recebimento de mensagem suspeita solicitando dados bancários e suspeita de tentativa de fraude.
3,7,"Meu cartao foi clonado e ja aparecem compras que nao fiz. Sou a cliente Fernanda Albuquerque, CPF 98765432100.","Meu cartao foi clonado e ja aparecem compras que nao fiz. Sou a cliente [NOME], CPF [CPF].",Cliente Fernanda Albuquerque relata que seu cartão foi clonado e que há compras não autorizadas em sua conta.,Cliente [NOME] relata que seu cartão foi clonado e que há compras não autorizadas em sua conta.
4,8,"Queria elogiar a rapidez na abertura da minha conta digital, atendente Julia Prado foi excelente.","Queria elogiar a rapidez na abertura da minha conta digital, atendente [NOME] foi excelente.",Cliente elogia a rapidez na abertura da conta digital e o excelente atendimento da colaboradora Julia Prado.,Cliente elogia a rapidez na abertura da conta digital e o excelente atendimento da colaboradora [NOME].
5,11,"Estou tentando falar com o suporte ha tres dias sobre um saque que nao caiu na minha conta 45231-9, agencia 0021.","Estou tentando falar com o suporte ha tres dias sobre um saque que nao caiu na minha conta [CONTA], agencia [CONTA].",Cliente reclama de falta de suporte há três dias para resolver problema de saque não creditado em sua conta.,Cliente reclama de falta de suporte há três dias para resolver problema de saque não creditado em sua conta.
6,13,"Preciso de uma segunda via do meu cartao, o antigo venceu. Meu nome completo e Maria da Silva Nogueira.","Preciso de uma segunda via do meu cartao, o antigo venceu. Meu nome completo e [NOME].",Cliente solicita segunda via de cartão de crédito devido ao vencimento do cartão antigo.,Cliente solicita segunda via de cartão de crédito devido ao vencimento do cartão antigo.
7,14,Foi descontado um valor de seguro que eu nunca contratei. Pode confirmar pelo meu e-mail joao.pereira@gmail.com?,Foi descontado um valor de seguro que eu nunca contratei. Pode confirmar pelo meu e-mail [EMAIL]?,Cliente reclama de desconto indevido de seguro nao contratado e solicita verificacao.,Cliente reclama de desconto indevido de seguro nao contratado e solicita verificacao.
8,17,Minha senha parou de funcionar e nao consigo acessar minha conta ha dois dias. RG 34.567.890-1.,Minha senha parou de funcionar e nao consigo acessar minha conta ha dois dias. RG [RG].,Cliente relata perda de acesso à conta devido a falha na senha há dois dias.,Cliente relata perda de acesso à conta devido a falha na senha há dois dias.
9,18,"Recebi um boleto de cobranca de um servico que ja cancelei ha meses. Assinado, rafael tavares moreira.","Recebi um boleto de cobranca de um servico que ja cancelei ha meses. Assinado, [NOME].","Cliente recebeu boleto de serviço cancelado há meses, reclama da cobrança indevida.","Cliente recebeu boleto de serviço cancelad

### 7.1 Visão de exportação
O que sai para a consultoria: só colunas anonimizadas e campos sem PII. O texto original não entra na exportação.

In [16]:
exportacao = pd.read_sql_query(
    '''SELECT id, tipo_manifestacao, sentimento, valor_citado, urgencia,
              texto_anonimizado AS texto, resumo_anonimizado AS resumo
       FROM manifestacoes WHERE status = 'processado' ORDER BY CAST(id AS INTEGER)''', con
)
exportacao.to_csv("exportacao_consultoria.csv", index=False)
exportacao.head(10)

,id,tipo_manifestacao,sentimento,valor_citado,urgencia,texto,resumo
0,1,reclamacao,negativo,NaN,alta,"Fui cobrado duas vezes na fatura deste mes, o valor total ja saiu da minha conta.","Cliente reclama de cobranca duplicada na fatura deste mes, com o valor total ja debitado de sua conta."
1,2,duvida,neutro,NaN,baixa,Gostaria de saber como altero meu endereco de cadastro no aplicativo.,Cliente tem dúvida sobre como alterar o endereço de cadastro no aplicativo do banco.
2,3,reclamacao,negativo,NaN,media,Meu nome e [NOME] e identifiquei uma transacao que nao reconheco no meu extrato de hoje.,"Cliente [NOME] relata transação desconhecida no extrato de hoje, solicitando esclarecimento."
3,4,elogio,positivo,NaN,baixa,"O atendimento da agencia foi otimo, o gerente [NOME] resolveu tudo rapido.",Cliente elogia o otimo atendimento da agencia e a rapidez do gerente [NOME] na resolucao de problemas.
4,5,fraude,negativo,NaN,alta,"Recebi uma mensagem estranha pedindo meus dados bancarios, acho que e uma fraude. Meu CPF e [CPF].",Cliente relata recebimento de mensagem suspeita solicitando dados bancários e suspeita de tentativa de fraude.
5,6,duvida,neutro,NaN,baixa,Quando sai a nova atualizacao do aplicativo?,Cliente pergunta quando será lançada a próxima atualização do aplicativo.
6,7,fraude,negativo,NaN,critica,"Meu cartao foi clonado e ja aparecem compras que nao fiz. Sou a cliente [NOME], CPF [CPF].",Cliente [NOME] relata que seu cartão foi clonado e que há compras não autorizadas em sua conta.
7,8,elogio,positivo,NaN,baixa,"Queria elogiar a rapidez na abertura da minha conta digital, atendente [NOME] foi excelente.",Cliente elogia a rapidez na abertura da conta digital e o excelente atendimento da colaboradora [NOME].
8,9,reclamacao,negativo,NaN,media,"Nao concordo com a taxa de manutencao cobrada este mes, parece um valor errado.","Cliente reclama da taxa de manutenção cobrada este mês, alegando que o valor está errado."
9,10,duvida,neutro,NaN,baixa,Como funciona o programa de pontos do cartao de credito?,Cliente tem dúvida sobre o funcionamento do programa de pontos do cartão de crédito.


## 8. Auditoria

### 8.1 Rotulagem manual

O dataset não tem gabarito. A rotulagem está em `rotulagem_manual.json`: cada id rotulado traz a lista de PII que existe no texto, com o tipo e o trecho exato. Os 40 ids foram rotulados (o mínimo pedido é 15). Ids **sem** PII ficam com lista vazia: eles entram na precisão, porque a função não pode mascarar texto comum.

Critérios usados na rotulagem (seção 2 do enunciado):

* Nome de qualquer pessoa citada, inclusive atendente, gerente e primeiro nome isolado (“A cliente Camila”).
* A palavra-chave antes do número (“CPF”, “conta”, “agência”) **não** é PII; só o número.
* Não são PII nesta atividade: final de cartão com 4 dígitos, valores, datas, telefone, nomes de produto (Pix) e cidades.

### 8.2 Métrica

Avaliação caixa-preta: usa só a saída de `anonimizar_texto`, como o teste do professor. Texto original e saída são tokenizados e alinhados com `difflib.SequenceMatcher`. Token do original que aparece num bloco igual do alinhamento foi preservado; o resto foi removido ou mascarado.

* **Recall** = tokens de PII que sumiram da saída ÷ total de tokens de PII rotulados.
* **Precisão** (definição do enunciado) = tokens não-PII que ficaram intactos ÷ total de tokens não-PII.

In [17]:
import difflib, re

_TOKEN = re.compile(r"[\w.+\-]+@[\w\-]+(?:\.[\w\-]+)+|\w+(?:[.\-/]\w+)*|[^\w\s]", re.UNICODE)


def tokenizar(texto: str):
    return [(m.group(0), m.start(), m.end()) for m in _TOKEN.finditer(texto)]


def avaliar_texto(texto: str, saida: str, rotulos: list[dict]) -> dict:
    tok_orig = tokenizar(texto)
    tok_out = [t for t, _, _ in tokenizar(saida)]
    sm = difflib.SequenceMatcher(a=[t for t, _, _ in tok_orig], b=tok_out, autojunk=False)
    preservado = [False] * len(tok_orig)
    for bloco in sm.get_matching_blocks():
        for k in range(bloco.a, bloco.a + bloco.size):
            preservado[k] = True

    spans = []
    for r in rotulos:
        ini = texto.find(r["texto"])
        if ini < 0:
            raise ValueError(f"rotulo nao encontrado: {r['texto']!r}")
        spans.append((ini, ini + len(r["texto"]), r["tipo"]))

    res = {"pii_total": 0, "pii_removido": 0, "nao_pii_total": 0, "nao_pii_intacto": 0, "fn": [], "fp": []}
    for (tok, a, b), ok in zip(tok_orig, preservado):
        tipo = next((t for (i, f, t) in spans if a >= i and b <= f), None)
        if tipo:
            res["pii_total"] += 1
            res["pii_removido"] += (not ok)
            if ok:
                res["fn"].append(f"{tok} ({tipo})")
        else:
            res["nao_pii_total"] += 1
            res["nao_pii_intacto"] += ok
            if not ok:
                res["fp"].append(tok)
    return res


def avaliar_conjunto(textos: dict, rotulos: dict, funcao) -> tuple[pd.DataFrame, float, float]:
    linhas = []
    for id_, rot in rotulos.items():
        saida = funcao(textos[id_])
        r = avaliar_texto(textos[id_], saida, rot)
        linhas.append({"id": id_, "saida": saida, **r})
    tab = pd.DataFrame(linhas)
    recall = tab["pii_removido"].sum() / tab["pii_total"].sum()
    precisao = tab["nao_pii_intacto"].sum() / tab["nao_pii_total"].sum()
    return tab, recall, precisao


rotulagem = json.loads(Path("rotulagem_manual.json").read_text(encoding="utf-8"))["rotulos"]
textos = dict(zip(df["id"], df["texto"]))
tab_texto, recall_texto, precisao_texto = avaliar_conjunto(textos, rotulagem, anonimizar_texto)
print(f"Campo TEXTO | ids rotulados: {len(rotulagem)} | tokens PII: {tab_texto.pii_total.sum()} "
      f"| recall: {recall_texto:.3f} | precisao: {precisao_texto:.3f}")
tab_texto[["id", "pii_total", "pii_removido", "fn", "fp", "saida"]]

Campo TEXTO | ids rotulados: 40 | tokens PII: 49 | recall: 1.000 | precisao: 1.000


,id,pii_total,pii_removido,fn,fp,saida
0,1,0,0,[],[],"Fui cobrado duas vezes na fatura deste mes, o valor total ja saiu da minha conta."
1,2,0,0,[],[],Gostaria de saber como altero meu endereco de cadastro no aplicativo.
2,3,3,3,[],[],Meu nome e [NOME] e identifiquei uma transacao que nao reconheco no meu extrato de hoje.
3,4,2,2,[],[],"O atendimento da agencia foi otimo, o gerente [NOME] resolveu tudo rapido."
4,5,1,1,[],[],"Recebi uma mensagem estranha pedindo meus dados bancarios, acho que e uma fraude. Meu CPF e [CPF]."
5,6,0,0,[],[],Quando sai a nova atualizacao do aplicativo?
6,7,3,3,[],[],"Meu cartao foi clonado e ja aparecem compras que nao fiz. Sou a cliente [NOME], CPF [CPF]."
7,8,2,2,[],[],"Queria elogiar a rapidez na abertura da minha conta digital, atendente [NOME] foi excelente."
8,9,0,0,[],[],"Nao concordo com a taxa de manutencao cobrada este mes, parece um valor errado."
9,10,0,0,[],[],Como funciona o programa de pontos do cartao de credito?


### 8.3 Auditoria do resumo gerado pela IA

O resumo pode reintroduzir um nome ou número que estava no texto. Duas checagens automáticas por id, usando a rotulagem do texto como referência:

* **Falso negativo no resumo:** algum trecho de PII rotulado no texto (ou parte de nome com 3+ letras, sem diferenciar maiúsculas) sobrevive em `resumo_anonimizado`.
* **Falso positivo no resumo:** aparece um marcador de um tipo que não existe na rotulagem daquele id (ex.: `[NOME]` num resumo de manifestação sem nome). Esses casos precisam de leitura manual para confirmar.

Os resumos mudam a cada execução do LLM, então esta tabela precisa ser lida **depois** da execução que gerou o `.db` entregue.

In [18]:
def pii_que_sobrou(resumo_anon: str, rotulos: list[dict]) -> list[str]:
    if not isinstance(resumo_anon, str) or not resumo_anon:  # NaN/None: revisao humana
        return []
    baixo = resumo_anon.lower()
    sobras = []
    for r in rotulos:
        partes = [r["texto"]] + ([p for p in re.findall(r"[^\W\d_]+", r["texto"]) if len(p) >= 3 and p.lower() not in {"das", "dos"}]
                                 if r["tipo"] == "NOME" else [re.sub(r"\D", "", r["texto"])])
        for p in partes:
            if p and (p.lower() in baixo or (r["tipo"] != "NOME" and p in re.sub(r"\D", "", resumo_anon) and len(p) >= 4)):
                sobras.append(f"{p} ({r['tipo']})")
                break
    return sobras


def marcadores_suspeitos(resumo_anon: str, rotulos: list[dict]) -> list[str]:
    if not isinstance(resumo_anon, str) or not resumo_anon:  # NaN/None: revisao humana
        return []
    tipos_ok = {r["tipo"] for r in rotulos}
    return [m for m in re.findall(r"\[(NOME|CPF|RG|CONTA|EMAIL)\]", resumo_anon) if m not in tipos_ok]


resumos = pd.read_sql_query("SELECT id, resumo, resumo_anonimizado FROM manifestacoes", con).set_index("id")
aud_resumo = []
for id_, rot in rotulagem.items():
    if id_ in resumos.index:
        ra = resumos.loc[id_, "resumo_anonimizado"]
        aud_resumo.append({"id": id_, "resumo": resumos.loc[id_, "resumo"], "resumo_anonimizado": ra,
                           "pii_sobrando": pii_que_sobrou(ra, rot),
                           "marcador_suspeito": marcadores_suspeitos(ra, rot)})
aud_resumo = pd.DataFrame(aud_resumo)
print("Resumos com PII sobrando:", (aud_resumo["pii_sobrando"].str.len() > 0).sum(),
      "| resumos com marcador suspeito:", (aud_resumo["marcador_suspeito"].str.len() > 0).sum())
aud_resumo[aud_resumo["resumo"] != aud_resumo["resumo_anonimizado"]]

Resumos com PII sobrando: 0 | resumos com marcador suspeito: 0


,id,resumo,resumo_anonimizado,pii_sobrando,marcador_suspeito
2,3,"Cliente Carlos Eduardo Ferreira relata transação desconhecida no extrato de hoje, solicitando esclarecimento.","Cliente [NOME] relata transação desconhecida no extrato de hoje, solicitando esclarecimento.",[],[]
3,4,Cliente elogia o otimo atendimento da agencia e a rapidez do gerente Bruno Salviano na resolucao de problemas.,Cliente elogia o otimo atendimento da agencia e a rapidez do gerente [NOME] na resolucao de problemas.,[],[]
6,7,Cliente Fernanda Albuquerque relata que seu cartão foi clonado e que há compras não autorizadas em sua conta.,Cliente [NOME] relata que seu cartão foi clonado e que há compras não autorizadas em sua conta.,[],[]
7,8,Cliente elogia a rapidez na abertura da conta digital e o excelente atendimento da colaboradora Julia Prado.,Cliente elogia a rapidez na abertura da conta digital e o excelente atendimento da colaboradora [NOME].,[],[]
30,31,Reclamação de cobrança indevida no cartão final 4521 por Pedro Henrique de Castro.,Reclamação de cobrança indevida no cartão final 4521 por [NOME].,[],[]
31,32,Solicita revisão do extrato dos últimos três meses da conta 998877-2.,Solicita revisão do extrato dos últimos três meses da conta [CONTA].,[],[]
33,34,"Cliente fornece contato: Thiago Bezerra Lins, email thiago.lins@empresa.com.br.","Cliente fornece contato: [NOME], email [EMAIL].",[],[]


### 8.4 Amostras antes/depois

Amostra fixa (seed) para leitura manual, incluindo ids sem PII, onde qualquer alteração seria falso positivo.

In [19]:
amostra = pd.read_sql_query(
    "SELECT id, texto, texto_anonimizado, resumo, resumo_anonimizado FROM manifestacoes", con
).sample(10, random_state=42).sort_values("id", key=lambda s: s.astype(int))
amostra

,id,texto,texto_anonimizado,resumo,resumo_anonimizado
4,5,"Recebi uma mensagem estranha pedindo meus dados bancarios, acho que e uma fraude. Meu CPF e 123.456.789-00.","Recebi uma mensagem estranha pedindo meus dados bancarios, acho que e uma fraude. Meu CPF e [CPF].",Cliente relata recebimento de mensagem suspeita solicitando dados bancários e suspeita de tentativa de fraude.,Cliente relata recebimento de mensagem suspeita solicitando dados bancários e suspeita de tentativa de fraude.
6,7,"Meu cartao foi clonado e ja aparecem compras que nao fiz. Sou a cliente Fernanda Albuquerque, CPF 98765432100.","Meu cartao foi clonado e ja aparecem compras que nao fiz. Sou a cliente [NOME], CPF [CPF].",Cliente Fernanda Albuquerque relata que seu cartão foi clonado e que há compras não autorizadas em sua conta.,Cliente [NOME] relata que seu cartão foi clonado e que há compras não autorizadas em sua conta.
12,13,"Preciso de uma segunda via do meu cartao, o antigo venceu. Meu nome completo e Maria da Silva Nogueira.","Preciso de uma segunda via do meu cartao, o antigo venceu. Meu nome completo e [NOME].",Cliente solicita segunda via de cartão de crédito devido ao vencimento do cartão antigo.,Cliente solicita segunda via de cartão de crédito devido ao vencimento do cartão antigo.
15,16,Gostaria de saber o horario de funcionamento da agencia central.,Gostaria de saber o horario de funcionamento da agencia central.,Cliente solicita informação sobre o horário de funcionamento da agência central.,Cliente solicita informação sobre o horário de funcionamento da agência central.
16,17,Minha senha parou de funcionar e nao consigo acessar minha conta ha dois dias. RG 34.567.890-1.,Minha senha parou de funcionar e nao consigo acessar minha conta ha dois dias. RG [RG].,Cliente relata perda de acesso à conta devido a falha na senha há dois dias.,Cliente relata perda de acesso à conta devido a falha na senha há dois dias.
19,20,"Adorei a nova funcao de investimentos automaticos, muito pratica.","Adorei a nova funcao de investimentos automaticos, muito pratica.","Cliente elogia a nova função de investimentos automáticos, achando-a muito prática.","Cliente elogia a nova função de investimentos automáticos, achando-a muito prática."
26,27,"Meu limite do cartao foi reduzido sem nenhum aviso previo. CPF 111.222.333-44, favor verificar.","Meu limite do cartao foi reduzido sem nenhum aviso previo. CPF [CPF], favor verificar.",Limite do cartão foi reduzido sem aviso prévio; cliente solicita verificação do problema.,Limite do cartão foi reduzido sem aviso prévio; cliente solicita verificação do problema.
27,28,Excelente iniciativa o novo programa de cashback.,Excelente iniciativa o novo programa de cashback.,"Cliente elogia o novo programa de cashback, considerando a iniciativa excelente.","Cliente elogia o novo programa de cashback, considerando a iniciativa excelente."
37,38,Gostaria de entender por que a fatura chegou com dois lancamentos duplicados.,Gostaria de entender por que a fatura chegou com dois lancamentos duplicados.,"Cliente questiona fatura com lançamentos duplicados, solicita esclarecimento.","Cliente questiona fatura com lançamentos duplicados, solicita esclarecimento."
39,40,Por favor me liguem no telefone cadastrado assim que possivel para tratar do estorno.,Por favor me liguem no telefone cadastrado assim que possivel para tratar do estorno.,Solicita contato telefônico imediato para tratar do estorno solicitado.,Solicita contato telefônico imediato para tratar do estorno solicitado.


### 8.5 Frases fora do conjunto público

A função não errou nenhuma PII nos 40 textos públicos nem nos resumos desta execução (seções 8.2 e 8.3). O enunciado pede pelo menos 2 casos de erro **reais**, encontrados rodando a função, e não hipotéticos. Para não inventar casos nem piorar a função de propósito, escrevemos 14 frases novas (`casos_fora_do_conjunto.json`), rotuladas com o mesmo critério, com formatos e contextos que não aparecem no CSV: pronome de tratamento popular (“Seu”, “Dona”), inicial solta, assinatura de setor, número de protocolo com hífen, agência com 2 dígitos.

Os erros abaixo saem de execução real da função sobre essas frases e são reproduzíveis. Eles entram no `casos_de_erro` do JSON com ids `extra_NN`, para deixar claro que não pertencem ao conjunto público. Os campos `recall_estimado` e `precisao_estimado` continuam sendo os medidos no conjunto público (seção 8.2), porque é sobre ele que o professor calcula o valor real.

Viés destas frases: foram escritas depois da função pronta, procurando pontos fracos. Então o número abaixo tende a ser pessimista, assim como o do conjunto público tende a ser otimista (as regras de contexto foram escritas olhando o CSV). O desempenho num conjunto novo deve ficar entre os dois.

In [20]:
extras = json.loads(Path("casos_fora_do_conjunto.json").read_text(encoding="utf-8"))["casos"]
textos_extra = {k: v["texto"] for k, v in extras.items()}
rotulos_extra = {k: v["rotulos"] for k, v in extras.items()}
tab_extra, recall_extra, precisao_extra = avaliar_conjunto(textos_extra, rotulos_extra, anonimizar_texto)
print(f"Fora do conjunto publico | frases: {len(extras)} | recall: {recall_extra:.3f} | precisao: {precisao_extra:.3f}")
tab_extra["texto"] = tab_extra["id"].map(textos_extra)
tab_extra[["id", "texto", "saida", "fn", "fp"]]

Fora do conjunto publico | frases: 14 | recall: 0.778 | precisao: 0.956


,id,texto,saida,fn,fp
0,extra_01,Estou falando com o Seu Jorge da agencia.,Estou falando com o Seu Jorge da agencia.,[Jorge (NOME)],[]
1,extra_02,Pedi para a Dona Cida ligar.,Pedi para a Dona Cida ligar.,[Cida (NOME)],[]
2,extra_03,"Grata, Silvia R.","Grata, [NOME] R.",[R (NOME)],[]
3,extra_04,"Att, Equipe Financeira","Att, [NOME]",[],"[Equipe, Financeira]"
4,extra_05,Quero falar com o Senhor Responsavel pelo setor.,Quero falar com o [NOME] pelo setor.,[],"[Senhor, Responsavel]"
5,extra_06,Minha reclamacao tem o protocolo numero 2024-5.,Minha reclamacao tem o protocolo numero [CONTA].,[],[2024-5]
6,extra_07,"Meu nome e Joao, conta 1234, agencia 56.","Meu nome e [NOME], conta [CONTA], agencia 56.",[56 (CONTA)],[]
7,extra_08,Falei com o atendente marcos na terca.,Falei com o atendente [NOME] na terca.,[],[]
8,extra_09,"Olá, sou a Mariana Lopes, cliente há 10 anos. Meu CPF é 321.654.987-10.","Olá, sou a [NOME], cliente há 10 anos. Meu CPF é [CPF].",[],[]
9,extra_10,"Transfiram para a conta 00012345-6 ag 0987, titular joão da silva.","Transfiram para a conta [CONTA] ag [CONTA], titular [NOME].",[],[]


### 8.6 Geração do `auditoria_resultado.json`

Os casos de erro vêm da execução real, nesta ordem: erros no texto do conjunto público, erros nos resumos, erros nas frases fora do conjunto (8.5). A célula não inventa casos.

**Revise as descrições antes do commit final**: elas são geradas a partir dos tokens e precisam ser lidas e, se necessário, reescritas com o que de fato aconteceu naquele id.

In [21]:
casos = []
for _, r in tab_texto.iterrows():
    if r["fn"]:
        casos.append({"id": r["id"], "tipo_erro": "falso_negativo",
                      "descricao": f"No campo texto, a funcao deixou passar {', '.join(r['fn'])}. Saida: {r['saida']}"})
    if r["fp"]:
        casos.append({"id": r["id"], "tipo_erro": "falso_positivo",
                      "descricao": f"No campo texto, a funcao mascarou por engano {', '.join(r['fp'])}. Saida: {r['saida']}"})
for _, r in aud_resumo.iterrows():
    if r["pii_sobrando"]:
        casos.append({"id": r["id"], "tipo_erro": "falso_negativo",
                      "descricao": f"No resumo gerado pela IA, sobrou {', '.join(r['pii_sobrando'])}. "
                                   f"Resumo anonimizado: {r['resumo_anonimizado']}"})
    if r["marcador_suspeito"]:
        casos.append({"id": r["id"], "tipo_erro": "falso_positivo",
                      "descricao": f"No resumo gerado pela IA, a funcao inseriu {', '.join('[' + m + ']' for m in r['marcador_suspeito'])} "
                                   f"num id sem esse tipo de PII (confirmar lendo o resumo). "
                                   f"Resumo: {r['resumo']} -> {r['resumo_anonimizado']}"})

for _, r in tab_extra.iterrows():
    origem = f"[Frase fora do conjunto publico, casos_fora_do_conjunto.json] Texto: {textos_extra[r['id']]} | Saida: {r['saida']} | "
    if r["fn"]:
        casos.append({"id": r["id"], "tipo_erro": "falso_negativo",
                      "descricao": origem + f"A funcao deixou passar {', '.join(r['fn'])}."})
    if r["fp"]:
        casos.append({"id": r["id"], "tipo_erro": "falso_positivo",
                      "descricao": origem + f"A funcao mascarou por engano: {', '.join(r['fp'])}."})

resultado = {
    "recall_estimado": round(float(recall_texto), 2),
    "precisao_estimado": round(float(precisao_texto), 2),
    "casos_de_erro": casos,
}
if len(casos) < 2:
    print(f"ATENCAO: so {len(casos)} caso(s) real(is) de erro. Nao invente casos.")
Path("auditoria_resultado.json").write_text(json.dumps(resultado, ensure_ascii=False, indent=2), encoding="utf-8")
print(json.dumps(resultado, ensure_ascii=False, indent=2))

{
  "recall_estimado": 1.0,
  "precisao_estimado": 1.0,
  "casos_de_erro": [
    {
      "id": "extra_01",
      "tipo_erro": "falso_negativo",
      "descricao": "[Frase fora do conjunto publico, casos_fora_do_conjunto.json] Texto: Estou falando com o Seu Jorge da agencia. | Saida: Estou falando com o Seu Jorge da agencia. | A funcao deixou passar Jorge (NOME)."
    },
    {
      "id": "extra_02",
      "tipo_erro": "falso_negativo",
      "descricao": "[Frase fora do conjunto publico, casos_fora_do_conjunto.json] Texto: Pedi para a Dona Cida ligar. | Saida: Pedi para a Dona Cida ligar. | A funcao deixou passar Cida (NOME)."
    },
    {
      "id": "extra_03",
      "tipo_erro": "falso_negativo",
      "descricao": "[Frase fora do conjunto publico, casos_fora_do_conjunto.json] Texto: Grata, Silvia R. | Saida: Grata, [NOME] R. | A funcao deixou passar R (NOME)."
    },
    {
      "id": "extra_04",
      "tipo_erro": "falso_positivo",
      "descricao": "[Frase fora do conjunto publi

### 8.7 Relatório de auditoria

In [22]:
# Fatos desta execucao usados no relatorio (calculados, nao digitados)
reintroduziu = aud_resumo[aud_resumo["resumo"] != aud_resumo["resumo_anonimizado"]]["id"].tolist()
print("Conjunto publico  -> recall:", round(recall_texto, 3), "| precisao:", round(precisao_texto, 3))
print("Fora do conjunto  -> recall:", round(recall_extra, 3), "| precisao:", round(precisao_extra, 3))
print(f"Resumos em que o LLM reintroduziu PII do texto: {len(reintroduziu)} de {len(resumos)} -> ids {reintroduziu}")
print("Resumos com PII sobrando depois da anonimizacao:", int((aud_resumo['pii_sobrando'].str.len() > 0).sum()))

Conjunto publico  -> recall: 1.0 | precisao: 1.0
Fora do conjunto  -> recall: 0.778 | precisao: 0.956
Resumos em que o LLM reintroduziu PII do texto: 7 de 40 -> ids ['3', '4', '7', '8', '31', '32', '34']
Resumos com PII sobrando depois da anonimizacao: 0


**Conjunto público.** Com os 40 ids rotulados, a função removeu toda a PII rotulada e não alterou nenhum token fora dela (célula acima). O resultado precisa ser lido com cuidado: as regras de contexto (“meu nome é”, “Att,”, “gerente”, palavra-chave antes de número) foram escritas olhando esse mesmo CSV. Os nomes em minúsculas (ids 18, 31 e 37), que seriam a limitação clássica do spaCy, foram cobertos justamente por essas regras. Por isso o número no conjunto público é otimista.

**Resumos.** O LLM reintroduziu no resumo nome, conta ou e-mail que estavam no texto em parte das manifestações (ids na célula acima). Em todos esses casos a função mascarou o que voltou. Isso mostra na prática por que o enunciado exige anonimizar o resumo também: se só o texto fosse anonimizado, a PII sairia pela coluna de resumo.

**Casos de erro (frases fora do conjunto público, seção 8.5).** Como a função não errou no conjunto público, os casos do JSON vêm de frases novas, com ids `extra_NN`:

* **extra_01 e extra_02, falso negativo.** “Seu Jorge” e “Dona Cida”: o NER não marcou esses nomes como pessoa e “Seu”/“Dona” não estão na lista de gatilhos. Pronome de tratamento popular antes do nome é um padrão que a função não cobre.
* **extra_03, falso negativo parcial.** “Silvia R.”: o nome foi mascarado, mas a inicial “R” ficou. A regra de extensão do nome só aceita palavras inteiras.
* **extra_04, falso positivo.** “Att, Equipe Financeira”: depois de “Att,” a função trata o que vem como assinatura de pessoa, sem checar se é nome de setor.
* **extra_05, falso positivo.** “Senhor Responsavel”: “Senhor” é gatilho e “Responsavel” começa com maiúscula, então virou [NOME].
* **extra_06, falso positivo.** “protocolo numero 2024-5”: o formato número-hífen-dígito é o mesmo de uma conta com dígito verificador, e a regra por formato não olha a palavra “protocolo” antes.
* **extra_07, falso negativo.** “agencia 56”: a regra de conta/agência exige pelo menos 3 dígitos, para não mascarar coisas como “conta de 20 reais”. Agência com 2 dígitos passa.

**Dependência do modelo.** Rodando a função sem o spaCy instalado (só as regras), o id 31 passa a falhar: “pedro henrique de castro” depois de “reclamacao de” não é pego, porque sem o NER a regra só aceita nome com maiúscula. Esse erro não entra no JSON, porque não acontece na configuração entregue, mas mostra quanto a função depende do NER.

## 9. Governança

### 9.1 Por que anonimizar localmente com spaCy e não com mais uma chamada de LLM

> **Confira os três motivos com o que foi discutido na Aula 13 e ajuste a redação se o professor usou outros termos.**

1. **Exposição do dado.** Mandar o texto original para um LLM externo para “remover a PII” é, em si, um novo compartilhamento de dado pessoal com um terceiro, geralmente com servidores fora do Brasil (transferência internacional, art. 33 da LGPD). Seria resolver a exportação para a consultoria criando outra exportação, justamente o problema que se quer evitar. Com spaCy e regex, o texto não sai da máquina do FIAP Bank.
2. **Previsibilidade e auditoria.** Regex e NER local dão sempre a mesma saída para a mesma entrada, podem ser testados com casos fixos e medidos (recall/precisão, seção 8). Um LLM varia entre execuções, pode resumir ou reescrever o texto em vez de só mascarar, e pode deixar passar ou até reintroduzir um nome sem aviso. Numa etapa de conformidade, precisamos conseguir explicar por que cada trecho foi mascarado.
3. **Custo, cota e disponibilidade.** A anonimização roda sobre todo o histórico, a cada exportação. Localmente não consome cota free-tier, não depende de rede nem do provedor estar no ar, e não passa pelo throttling. A pilha de resiliência da seção 5 existe porque as APIs falham; a etapa de LGPD não deve depender delas.

### 9.2 Por que o dado original continua no banco

* **Atendimento.** A manifestação precisa ser respondida ao cliente certo. Sem nome, CPF ou conta, o banco não consegue localizar a transação contestada, estornar valor ou tratar uma fraude.
* **Auditoria e prestação de contas.** A LGPD pede que o controlador registre as operações de tratamento (art. 37) e consiga demonstrar as medidas adotadas (art. 6º, X). A própria auditoria da seção 8 só é possível comparando o original com a versão anonimizada.
* **Direitos do titular.** O cliente pode pedir acesso, correção ou informação sobre o uso dos seus dados (art. 18). Isso exige o dado identificado.
* **Anonimização é irreversível e pode errar.** Se a função tiver um erro (seção 8), o original permite corrigir a função e gerar de novo a versão anonimizada. Se só existisse a versão mascarada, o erro ficaria permanente, ou pior, a PII que vazou não teria como ser localizada.

A separação é por finalidade: o original fica no banco interno com acesso restrito, e só a visão de exportação (seção 7.1), com as colunas anonimizadas, sai para a consultoria. Dado anonimizado deixa de ser dado pessoal (art. 12) justamente porque não carrega o vínculo com o titular; o banco interno continua sendo dado pessoal e continua sujeito a todas as obrigações da LGPD.

In [23]:
con.close()